<a href="https://colab.research.google.com/github/arnolosabas/Seminario-de-Investigacion-/blob/main/C%C3%B3digo_de_Seminario_de_Investigaci%C3%B3n.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import networkx as nx
import numpy as np

In [3]:
def laplaciano(G):
    L = nx.laplacian_matrix(G).toarray()
    return L.astype(float)

def espectro_laplaciano(G):
    L = laplaciano(G)
    valores = np.linalg.eigvalsh(L)
    valores[np.abs(valores) < 1e-12] =0
    return valores

def espectro_densidad(G):
    valores = espectro_laplaciano(G)
    return valores / np.sum(valores)

def entropia_von_neumann(G):
    p = espectro_densidad(G)
    p = p[p > 1e-12]
    S = -np.sum(p * np.log2(p))
    return S

In [4]:
def codigo_arbol(T):
    return nx.to_graph6_bytes(
        T,
        header=False
    ).decode().strip()

def analizar_arbol(T):
    espectro = espectro_laplaciano(T)
    entropia = entropia_von_neumann(T)

    grados = [d for v, d in T.degree()]
    grados.sort(reverse=True)

    datos = {
        "n": T.number_of_nodes(),
        "entropia": entropia,
        "espectro": espectro,
        "grados": grados,
        "grado_maximo": max(grados),
        "hojas": sum(d == 1 for d in grados),
        "diametro": nx.diameter(T),
        "radio": nx.radius(T),
        "codigo": codigo_arbol(T)
    }

    return datos

def son_cospectrales(datos1, datos2, tol=1e-10):
    return np.allclose(
        datos1,
        datos2,
        atol=tol,
        rtol=0
    )
    #arreglar

def son_coentropicos(datos1, datos2, tol=1e-10):
    return abs(
        datos1 - datos2
    ) < tol

    #arreglar

In [6]:
arboles = list(nx.nonisomorphic_trees(11))
for i in range(len(arboles)):
    for j in range(i + 1, len(arboles)):
        u = arboles[i]
        v = arboles[j]
        if son_coentropicos(entropia_von_neumann(u), entropia_von_neumann(v)):
          print("Candidatos a coentropicos:", i, j)
        if son_cospectrales(espectro_laplaciano(u), espectro_laplaciano(v)):
          print("Candidatos a cospectrales:", i, j)

Candidatos a coentropicos: 15 24
Candidatos a cospectrales: 15 24
Candidatos a coentropicos: 37 51
Candidatos a cospectrales: 37 51
Candidatos a coentropicos: 97 117
Candidatos a cospectrales: 97 117
